In [1]:
import pandas as pd
import numpy as np
import pickle
from pathlib import Path
from scipy.sparse import csr_matrix, save_npz

In [2]:
def preprocess_views_data_sparse(input_csv, dataset_name="MyDataset",
                                 min_interactions=5, max_interactions=100, train_ratio=0.8):
    # Load data
    df = pd.read_csv(input_csv, sep=",")
    df = df[["TIMESTAMP", "USER_ID", "ARTICLE_ID"]]

    # Try to convert TIMESTAMP → datetime; if fails, keep as string for sorting
    try:
        df["TIMESTAMP"] = pd.to_datetime(df["TIMESTAMP"], errors="raise")
    except Exception:
        print("⚠️ TIMESTAMP not in datetime format, treating as string (sorted lexicographically).")
        df["TIMESTAMP"] = df["TIMESTAMP"].astype(str)

    # Filter users by interaction count
    user_groups = df.groupby("USER_ID")
    valid_users = [u for u, g in user_groups if min_interactions <= len(g) <= max_interactions]
    df = df[df["USER_ID"].isin(valid_users)].copy()   # ✅ avoid SettingWithCopyWarning

    # Re-map users and items to integer indices
    unique_users = df["USER_ID"].unique()
    unique_items = df["ARTICLE_ID"].unique()

    user2id = {u: i for i, u in enumerate(unique_users)}
    id2user = {i: u for u, i in user2id.items()}

    item2id = {it: i for i, it in enumerate(unique_items)}
    id2item = {i: it for it, i in item2id.items()}

    df["user_idx"] = df["USER_ID"].map(user2id)
    df["item_idx"] = df["ARTICLE_ID"].map(item2id)

    # Train/test split per user
    train_interactions, test_interactions = [], []
    for u, g in df.groupby("user_idx"):
        g = g.sort_values("TIMESTAMP")
        n_train = int(len(g) * train_ratio)
        train_interactions.append(g.iloc[:n_train])
        test_interactions.append(g.iloc[n_train:])

    train_df = pd.concat(train_interactions)
    test_df = pd.concat(test_interactions)

    num_users = len(unique_users)
    num_items = len(unique_items)

    # Build sparse train/test matrices
    train_matrix = csr_matrix(
        (np.ones(len(train_df)), (train_df["user_idx"], train_df["item_idx"])),
        shape=(num_users, num_items),
        dtype=np.int8
    )

    test_matrix = csr_matrix(
        (np.ones(len(test_df)), (test_df["user_idx"], test_df["item_idx"])),
        shape=(num_users, num_items),
        dtype=np.int8
    )

    # Build static test data (CSR matrix + pos/neg columns)
    pos_items, neg_items = [], []
    rng = np.random.default_rng(42)

    for u in range(num_users):
        user_pos_items = test_df[test_df.user_idx == u]["item_idx"].tolist()
        if not user_pos_items:  # no test items
            pos_items.append(-1)
            neg_items.append(-1)
            continue
        pos_item = rng.choice(user_pos_items)
        neg_candidates = list(set(range(num_items)) -
                              set(train_df[train_df.user_idx == u]["item_idx"]) -
                              set(user_pos_items))
        if not neg_candidates:
            neg_item = rng.integers(0, num_items)
        else:
            neg_item = rng.choice(neg_candidates)

        pos_items.append(pos_item)
        neg_items.append(neg_item)

    pos_items = np.array(pos_items).reshape(-1, 1)
    neg_items = np.array(neg_items).reshape(-1, 1)

    # Popularity dictionary
    pop_dict = train_df["item_idx"].value_counts().to_dict()

    # Save outputs
    out_path = Path(".") / dataset_name
    out_path.mkdir(parents=True, exist_ok=True)

    save_npz(out_path / f"train_data_{dataset_name}.npz", train_matrix)
    save_npz(out_path / f"test_data_{dataset_name}.npz", test_matrix)
    np.savez(out_path / f"static_test_data_{dataset_name}.npz",
             data=test_matrix, pos=pos_items, neg=neg_items)

    with open(out_path / f"pop_dict_{dataset_name}.pkl", "wb") as f:
        pickle.dump(pop_dict, f)

    with open(out_path / f"user_mapping_{dataset_name}.pkl", "wb") as f:
        pickle.dump({"user2id": user2id, "id2user": id2user}, f)

    with open(out_path / f"item_mapping_{dataset_name}.pkl", "wb") as f:
        pickle.dump({"item2id": item2id, "id2item": id2item}, f)

    print(f"✅ Preprocessing complete. Files saved to {out_path}")

In [3]:
# Run preprocessing
preprocess_views_data_sparse("views_data.csv", dataset_name="Nu.nl_reads")

✅ Preprocessing complete. Files saved to Nu.nl_reads
